# PDF to Chroma Pipeline

This notebook loads a PDF, splits it into chunks, stores the chunks in Chroma, and runs a retrieval query.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Uttam\AppData\Local\Temp\ipykernel_8432\1126683846.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Prepare Paths and Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('e:/Projects/Campusx-Advance-Rag/Code/Advanced_Rag_Codes/04_vector_stores')

In [6]:
env_path = project_root / ".env"
pdf_path = project_root / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"
collection_name = "rag-pipeline"

print(f"PDF path: {pdf_path}")
print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

PDF path: e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
Collection name: rag-pipeline
Persist directory: e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\04_vector_stores\notebooks\chroma_langchain_db


In [7]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not HF_TOKEN:
    raise ValueError("Please add your HF_TOKEN to the .env file.")

In [8]:
# Reuse the same embedding model as the other notebooks in this repo.
embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-small-en-v1.5",
    huggingfacehub_api_token=HF_TOKEN
)
print("Embedding model is ready.")

Embedding model is ready.


## 2. Add Small Display Helpers

In [9]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Load the PDF

In [10]:
loader = PyPDFLoader(str(pdf_path))

In [11]:
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

incorrect startxref pointer(1)
parsing for Object Streams


Total pages loaded: 6


In [14]:
print(docs[0].page_content)
print()
print(docs[0].metadata)

Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, tools, and retrieval in modern AI systems
By Editorial Desk
The moment AI stopped feeling like a demo
For a long time, the most common experience with AI felt theatrical. You typed a question, the model
answered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a
follow-up that required memory, factual grounding, or a small sequence of actions, and the system often fell
apart. It could sound confident without being connected to anything real. That gap between fluency and
usefulness is exactly where AI agents enter the picture.
An AI agent is interesting not because it sounds human, but because it behaves like software with intent. It
can take a goal, figure out what it needs in order to make progress, and work through a sequence of steps
instead of improvising a single reply. In the simplest form, that might mean searching a f

In [13]:
print(f"First page preview: {preview_text(docs[0].page_content)}")
print(f"First page metadata: {docs[0].metadata}")

First page preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First page metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'e:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\04_vector_stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 4. Split the PDF into Chunks

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

In [16]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 88


In [17]:
print(f"First chunk preview: {preview_text(chunked_docs[0].page_content)}")
print(f"First chunk metadata: {chunked_docs[0].metadata}")

First chunk preview: Page 1
 Beyond Chatbots: Why AI Agents Feel Like the
 Next Real Shift
A practical long-form blog on planning, memory, to...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T20:36:07+05:00', 'author': 'By Editorial Desk', 'keywords': '', 'moddate': '2026-03-12T20:36:07+05:00', 'subject': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'trapped': '/False', 'source': 'e:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\04_vector_stores\\documents\\beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


## 5. Store Chunks in Chroma

In [18]:
collection_name

'rag-pipeline'

In [19]:
persist_directory

WindowsPath('e:/Projects/Campusx-Advance-Rag/Code/Advanced_Rag_Codes/04_vector_stores/notebooks/chroma_langchain_db')

In [20]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory),
)

print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

Stored 88 chunks in the 'rag-pipeline' collection.


## 6. Retrieve Relevant Chunks

In [21]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [22]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

Query: How do AI agents use tools and memory?

Retrieved chunks:
1. page=2 | source=e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
2. page=0 | source=e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\04_vector_stores\documents\beyond-chatbots-ai-agents-next-real-shift.pdf
   content=participate in real workflows.
This is also why vector stores and retrieval have become such central topics in modern AI tutorials. Once you
accept that an agent should gather context instead of guessing from memory alone, you need a mechanism to
3. page=5 | source=e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Co

In [23]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print(f"page_no. {doc.metadata.get("page_label")}")
    print()

Score: 0.3405
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
page_no. 3

Score: 0.4231
Content preview: participate in real workflows.
This is also why vector stores and retrieval have become such central topics in modern AI tutorials. Once you
accept that an agent should gather context instead of guessing from memory alone, you need a mechanism to
page_no. 1

